# BugInsight AI — Google Colab Baseline Training Notebook

**Target hardware:** Google Colab GPU runtime (T4 / A100 / L4)  
**Purpose:** Fully automated, headless cloud training for the E1 multi-seed experiment suite.  

This notebook trains all four baseline models (Random Forest, XGBoost, BiLSTM, CodeBERT)  
across five random seeds, generates Table 1 with statistical comparisons, and saves  
all outputs to Google Drive for persistent storage.

| Model | Type | Seeds |
|---|---|---|
| Random Forest | Classical ML (TF-IDF) | 42, 123, 456, 789, 999 |
| XGBoost | Classical ML (TF-IDF) | 42, 123, 456, 789, 999 |
| BiLSTM | Deep Learning | 42, 123, 456, 789, 999 |
| CodeBERT | Transformer (Fine-tuned) | 42, 123, 456, 789, 999 |

## 0. Mount Google Drive

Mount Google Drive so that all outputs can be persisted beyond the Colab session lifetime.

In [ ]:
"""Step 0 — Mount Google Drive for persistent storage."""
from google.colab import drive

drive.mount("/content/drive")

# Create a project output folder on Drive
import os

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/BugInsight_AI_Results"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f"Google Drive mounted. Outputs will be saved to: {DRIVE_OUTPUT_DIR}")

## 1. Environment Check

Verify GPU availability, CUDA version, and system memory before training.

In [ ]:
"""Step 1 — Environment Check: GPU info, CUDA version, memory."""
import subprocess
import os
import sys
import platform

print("=" * 70)
print("BugInsight AI — Environment Check (Google Colab)")
print("=" * 70)
print(f"Python     : {sys.version}")
print(f"Platform   : {platform.platform()}")
print()

# --- GPU / CUDA info via nvidia-smi ---
try:
    result = subprocess.run(
        ["nvidia-smi"],
        capture_output=True, text=True, check=True,
    )
    print(result.stdout)
except FileNotFoundError:
    print("WARNING: nvidia-smi not found — no GPU detected.")
    print("TIP: Go to Runtime → Change runtime type → GPU")

# --- PyTorch GPU probe ---
try:
    import torch
    print(f"PyTorch    : {torch.__version__}")
    print(f"CUDA avail : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA ver   : {torch.version.cuda}")
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            mem_gb = props.total_mem / (1024 ** 3)
            print(f"  GPU {i}    : {props.name} ({mem_gb:.1f} GB)")
except ImportError:
    print("PyTorch not yet installed — will install in step 3.")

# --- System memory ---
try:
    with open("/proc/meminfo", "r") as f:
        for line in f:
            if line.startswith(("MemTotal", "MemAvailable")):
                print(line.strip())
except FileNotFoundError:
    print("(Cannot read /proc/meminfo — non-Linux host.)")

# --- Disk space ---
try:
    result = subprocess.run(["df", "-h", "/"], capture_output=True, text=True)
    print("\nDisk space:")
    print(result.stdout)
except FileNotFoundError:
    pass

print("=" * 70)

## 2. Clone Repository

Clone the BugInsight AI repository from GitHub into `/content/buginsight-ai`  
and set it as the working directory.

In [ ]:
"""Step 2 — Clone Repository."""
import os

REPO_URL = "https://github.com/YOUR_USERNAME/buginsight-ai.git"
CLONE_DIR = "/content/buginsight-ai"

if os.path.isdir(CLONE_DIR):
    print(f"Repository already exists at {CLONE_DIR} — pulling latest changes.")
    !cd {CLONE_DIR} && git pull --ff-only
else:
    print(f"Cloning {REPO_URL} ...")
    !git clone {REPO_URL} {CLONE_DIR}

%cd {CLONE_DIR}
print(f"\nWorking directory: {os.getcwd()}")
!ls -la

## 3. Install Dependencies

Install all Python dependencies. Colab comes with most ML packages  
pre-installed, so this mainly picks up any missing or version-specific packages.

In [ ]:
"""Step 3 — Install Dependencies."""
!pip install -r requirements.txt -q

# Verify critical imports
import torch
import transformers
import sklearn
import xgboost
import yaml

print("\n" + "=" * 70)
print("Dependency verification:")
print(f"  torch          : {torch.__version__}")
print(f"  transformers   : {transformers.__version__}")
print(f"  scikit-learn   : {sklearn.__version__}")
print(f"  xgboost        : {xgboost.__version__}")
print(f"  pyyaml         : {yaml.__version__}")
print(f"  CUDA available : {torch.cuda.is_available()}")
print("=" * 70)

## 4. Generate Synthetic Data

Generate a small synthetic dataset (~2,000 samples) for **smoke testing**.  
For real research, replace this step with your actual Bugzilla/Defects4J data.

In [ ]:
"""Step 4 — Generate Synthetic Data for smoke testing."""
!python scripts/generate_synthetic_data.py

# Quick verification
import pandas as pd

df = pd.read_csv("data/raw/synthetic_bugs.csv")
print(f"\nSynthetic dataset shape: {df.shape}")
print(f"\nSeverity distribution:")
print(df["severity"].value_counts().to_string())
print(f"\nSample record:")
print(df.iloc[0].to_string())

## 5. Preprocess Data

Run the data preprocessing pipeline: standardize severity labels, clean text,  
and generate train/validation/test splits.

In [ ]:
"""Step 5 — Preprocess Data."""
!python -m data.preprocess

# Verify processed output exists
import os

processed_dir = "data/processed"
if os.path.isdir(processed_dir):
    print(f"\nProcessed files in {processed_dir}:")
    for fname in sorted(os.listdir(processed_dir)):
        fpath = os.path.join(processed_dir, fname)
        size_kb = os.path.getsize(fpath) / 1024
        print(f"  {fname:40s} {size_kb:8.1f} KB")
else:
    print(f"WARNING: {processed_dir} not found — preprocessing may have failed.")

## 6. Run All Experiments

Train all four baseline models across five random seeds (20 runs total):  
- **Random Forest** (TF-IDF + sklearn)  
- **XGBoost** (TF-IDF + gradient boosting)  
- **BiLSTM** (learned embeddings + bidirectional LSTM)  
- **CodeBERT** (fine-tuned `microsoft/codebert-base`)  

This is the most time-consuming step — expect ~30-60 minutes on a T4 GPU.

In [ ]:
"""Step 6 — Run All Multi-Seed Experiments."""
import time

start_time = time.time()
!python scripts/run_all_experiments.py
elapsed = time.time() - start_time

hours, remainder = divmod(int(elapsed), 3600)
minutes, seconds = divmod(remainder, 60)
print(f"\nTotal training time: {hours:02d}h {minutes:02d}m {seconds:02d}s")

## 7. Generate Table 1

Skip training and regenerate Table 1 from the saved metrics.  
This also runs the CodeBERT-vs-baseline statistical significance tests  
(paired t-test and Wilcoxon signed-rank test).

In [ ]:
"""Step 7 — Generate Table 1 (skip training, stats only)."""
!python scripts/run_all_experiments.py --skip-training

## 8. Display Results

Read all per-seed metric JSON files from `outputs/metrics/` and build a  
comprehensive summary table with mean ± std across seeds.

In [ ]:
"""Step 8 — Display Results: read metrics JSONs and print summary table."""
import json
import os
from collections import defaultdict
from pathlib import Path

import numpy as np

metrics_dir = Path("outputs/metrics")
if not metrics_dir.is_dir():
    print(f"ERROR: {metrics_dir} does not exist. Did training complete?")
else:
    # Collect all metric files
    model_metrics: dict = defaultdict(list)
    metric_files = sorted(metrics_dir.glob("*.json"))

    print(f"Found {len(metric_files)} metric file(s) in {metrics_dir}\n")

    for fpath in metric_files:
        with open(fpath, "r") as f:
            data = json.load(f)

        # Extract model name from filename (e.g., 'random_forest_seed42.json')
        fname = fpath.stem
        # Handle filenames like 'codebert_seed42' or 'random_forest_seed789'
        parts = fname.rsplit("_seed", 1)
        model_name = parts[0] if len(parts) == 2 else fname

        model_metrics[model_name].append(data)

    # Build summary table
    KEY_METRICS = ["accuracy", "precision", "recall", "f1", "macro_f1", "weighted_f1"]

    print("=" * 90)
    print("BugInsight AI — Multi-Seed Results Summary (Table 1)")
    print("=" * 90)
    header = f"{'Model':<20s}"
    for m in KEY_METRICS:
        header += f"  {m:>14s}"
    header += f"  {'Seeds':>6s}"
    print(header)
    print("-" * 90)

    for model_name in ["random_forest", "xgboost", "bilstm", "codebert"]:
        records = model_metrics.get(model_name, [])
        if not records:
            print(f"{model_name:<20s}  (no metrics found)")
            continue

        row = f"{model_name:<20s}"
        for metric_key in KEY_METRICS:
            values = [
                r[metric_key] for r in records if metric_key in r
            ]
            if values:
                mean_val = np.mean(values)
                std_val = np.std(values)
                row += f"  {mean_val:6.4f}±{std_val:.4f}"
            else:
                row += f"  {'N/A':>14s}"
        row += f"  {len(records):>6d}"
        print(row)

    print("=" * 90)

    # Print per-seed detail for the best model (CodeBERT)
    if "codebert" in model_metrics:
        print("\n--- CodeBERT Per-Seed Detail ---")
        for i, rec in enumerate(model_metrics["codebert"]):
            seed = rec.get("seed", f"run_{i}")
            f1 = rec.get("f1", rec.get("macro_f1", "N/A"))
            acc = rec.get("accuracy", "N/A")
            print(f"  Seed {seed}: F1={f1}, Accuracy={acc}")

    # Check for Table 1 figure files
    figures_dir = Path("outputs/figures")
    if figures_dir.is_dir():
        table_files = list(figures_dir.glob("table1*"))
        if table_files:
            print(f"\nTable 1 artifact(s) saved:")
            for tf in table_files:
                print(f"  {tf}")

## 9. Save Outputs to Google Drive

Zip all outputs (metrics, models, figures, logs) and copy them to Google Drive  
for persistent storage beyond the Colab session.

In [ ]:
"""Step 9 — Save Outputs to Google Drive."""
import shutil
import os
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
archive_name = f"buginsight_results_{timestamp}"
output_src = "outputs"

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/BugInsight_AI_Results"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

if os.path.isdir(output_src):
    # --- 1. Create local zip archive ---
    local_archive = f"/content/{archive_name}"
    shutil.make_archive(local_archive, "zip", ".", output_src)
    local_zip = f"{local_archive}.zip"

    # --- 2. Copy zip to Google Drive ---
    drive_zip = os.path.join(DRIVE_OUTPUT_DIR, f"{archive_name}.zip")
    shutil.copy2(local_zip, drive_zip)

    # --- 3. Also copy raw outputs directory to Drive ---
    drive_outputs = os.path.join(DRIVE_OUTPUT_DIR, f"outputs_{timestamp}")
    shutil.copytree(output_src, drive_outputs, dirs_exist_ok=True)

    zip_size_mb = os.path.getsize(drive_zip) / (1024 ** 2)

    print("=" * 70)
    print("BugInsight AI — Outputs Saved to Google Drive")
    print("=" * 70)
    print(f"Archive    : {drive_zip}")
    print(f"Raw outputs: {drive_outputs}")
    print(f"Zip size   : {zip_size_mb:.2f} MB")
    print()

    # List contents summary
    for dirpath, dirnames, filenames in os.walk(output_src):
        level = dirpath.replace(output_src, "").count(os.sep)
        indent = "  " * level
        print(f"{indent}{os.path.basename(dirpath)}/")
        sub_indent = "  " * (level + 1)
        for fname in sorted(filenames):
            fpath = os.path.join(dirpath, fname)
            fsize = os.path.getsize(fpath) / 1024
            print(f"{sub_indent}{fname} ({fsize:.1f} KB)")

    print(f"\nAll outputs are safely stored on Google Drive at:")
    print(f"  {DRIVE_OUTPUT_DIR}")
    print("=" * 70)
else:
    print(f"ERROR: '{output_src}' directory not found. No outputs to archive.")